# **SESIÓN 3:** Autocorrelación y Fundamentos de Pronóstico
## Universidad Autónoma de Occidente
### Maestría en Inteligencia Artificial y Ciencias de Datos
**Instructor:** Dr. Ing. Sergio A. Cantillo · sacantillo@uao.edu.co

---

### 🎯 Objetivos de la Sesión
- Detectar y tratar **problemas de calidad de datos** antes del modelado
- Comprender y aplicar la **ACF** como herramienta de diagnóstico
- Aplicar el **Test de Ljung-Box** para verificar estructura temporal
- Implementar y comparar **4 modelos baseline** con `statsforecast`
- Evaluar pronósticos con métricas formales usando `utilsforecast`
- Ejecutar **Time Series Cross-Validation** temporal

### 📦 Dataset
| Dataset | Área | Descripción |
|---------|------|-------------|
| **Perrin Frères — Champagne** | 🍾 Retail | Ventas mensuales 1964–1972 (108 obs.) |

---

## ⚙️ PARTE 1: Configuración del Entorno

In [1]:
!pip install statsforecast utilsforecast statsmodels -q

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, WindowAverage, RandomWalkWithDrift
from utilsforecast.losses import mae, rmse, smape
from utilsforecast.evaluation import evaluate
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("✅ Todas las librerías cargadas correctamente")

✅ Todas las librerías cargadas correctamente


/home/alejo/.virtualenvs/datascience-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 📦 PARTE 2: Carga de Datos y Formato Nixtla

Los datos inicialmente tienen el reporte individual de los casos de dengue diariamente, sin embargo el análisis de los datos desde el dominio de la salud se hace por semanas, de esta manera se hace el control y la vigilancia epidemiológica.

Lo cargamos directamente y lo convertimos al formato estándar `unique_id | ds | y`.


In [3]:
import pandas as pd

# =========================
# 1. Cargar dataset
# =========================

#df = pd.read_csv("/content/datos_dengue_202604282048.csv")
df = pd.read_csv("datos_dengue.csv")

print(df.head())
print(df.columns)

# =========================
# 2. Convertir fecha
# =========================

df["fec_not"] = pd.to_datetime(df["fec_not"], errors="coerce")

# eliminar nulos
df = df.dropna(subset=["fec_not"])

# =========================
# 3. Agrupar por año + semana
# =========================

# sacar año y semana ISO
df["anio"] = df["fec_not"].dt.isocalendar().year
df["semana"] = df["fec_not"].dt.isocalendar().week

# crear etiqueta tipo: 2023-W15
df["anio_semana"] = (
    df["anio"].astype(str)
    + "-W" +
    df["semana"].astype(str).str.zfill(2)
)

# conteo semanal por año
conteo_semanal = (
    df.groupby("anio_semana")
    .size()
    .reset_index(name="y")
)

# formato Nixtla
conteo_semanal["unique_id"] = "dengue_cali"

conteo_semanal = conteo_semanal.rename(columns={
    "anio_semana": "ds"
})

conteo_semanal = conteo_semanal[
    ["unique_id", "ds", "y"]
]

print(conteo_semanal.head(20))

# =========================
# 4. Guardar resultado
# =========================

conteo_semanal.to_csv(
    "dengue_anio_semana_nixtla.csv",
    index=False
)

print("Archivo generado correctamente")
print("dengue_anio_semana_nixtla.csv")

      fec_not  semana
0  2010-12-03      48
1  2010-02-25       7
2  2010-01-16       1
3  2010-05-24      19
4  2010-03-26      11
Index(['fec_not', 'semana'], dtype='object')
      unique_id        ds    y
0   dengue_cali  2009-W53   12
1   dengue_cali  2010-W01  142
2   dengue_cali  2010-W02  210
3   dengue_cali  2010-W03  253
4   dengue_cali  2010-W04  345
5   dengue_cali  2010-W05  375
6   dengue_cali  2010-W06  504
7   dengue_cali  2010-W07  573
8   dengue_cali  2010-W08  543
9   dengue_cali  2010-W09  475
10  dengue_cali  2010-W10  451
11  dengue_cali  2010-W11  439
12  dengue_cali  2010-W12  384
13  dengue_cali  2010-W13  304
14  dengue_cali  2010-W14  330
15  dengue_cali  2010-W15  308
16  dengue_cali  2010-W16  293
17  dengue_cali  2010-W17  211
18  dengue_cali  2010-W18  227
19  dengue_cali  2010-W19  234
Archivo generado correctamente
dengue_anio_semana_nixtla.csv


In [4]:
import pandas as pd

RUTA_DATOS = "datos_dengue.csv"
RUTA_SALIDA = "dengue_semanal_nixtla.csv"

def display_dataframe(title: str, dataframe: pd.DataFrame, n: int | None = None) -> None:
    """Imprime un título y despliega un dataframe."""
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")
    display(dataframe if n is None else dataframe.head(n))

def cargar_datos_dengue(ruta_archivo):
    df = pd.read_csv(ruta_archivo)
    df["fec_not"] = pd.to_datetime(df["fec_not"], errors="coerce")
    df = df.dropna(subset=["fec_not"]).copy()
    return df


def convertir_a_serie_semanal_nixtla(
    df,
    columna_fecha="fec_not",
    unique_id="dengue_cali"
):
    serie = (
        df.assign(
            ds=df[columna_fecha]
            - pd.to_timedelta(df[columna_fecha].dt.weekday, unit="D")
        )
        .groupby("ds", as_index=False)
        .size()
        .rename(columns={"size": "y"})
        .sort_values("ds")
    )

    serie["unique_id"] = unique_id
    serie = serie[["unique_id", "ds", "y"]]

    return serie


df_crudo = cargar_datos_dengue(RUTA_DATOS)

df = convertir_a_serie_semanal_nixtla(df_crudo)

print(df.head(10))
print(df.tail(10))

print("\nFilas:", len(df))
print("Fechas únicas:", df["ds"].nunique())
print("Duplicados unique_id-ds:", df.duplicated(["unique_id", "ds"]).sum())

df.to_csv(RUTA_SALIDA, index=False)

print(f"\nArchivo generado correctamente: {RUTA_SALIDA}")

     unique_id         ds    y
0  dengue_cali 2009-12-28   12
1  dengue_cali 2010-01-04  142
2  dengue_cali 2010-01-11  210
3  dengue_cali 2010-01-18  253
4  dengue_cali 2010-01-25  345
5  dengue_cali 2010-02-01  375
6  dengue_cali 2010-02-08  504
7  dengue_cali 2010-02-15  573
8  dengue_cali 2010-02-22  543
9  dengue_cali 2010-03-01  475
       unique_id         ds    y
731  dengue_cali 2024-01-01  213
732  dengue_cali 2024-01-08   42
733  dengue_cali 2024-01-15   13
734  dengue_cali 2024-01-22    3
735  dengue_cali 2024-01-29    6
736  dengue_cali 2024-02-05    4
737  dengue_cali 2024-02-12    7
738  dengue_cali 2024-02-19    6
739  dengue_cali 2024-03-04    5
740  dengue_cali 2024-03-25    1

Filas: 741
Fechas únicas: 741
Duplicados unique_id-ds: 0

Archivo generado correctamente: dengue_semanal_nixtla.csv


In [5]:
# Visualización de la serie completa
import pandas as pd
import plotly.graph_objects as go

# =========================
# 1. Cargar archivo semanal
# =========================

#df = pd.read_csv("/content/dengue_anio_semana_nixtla.csv")

#print(df.head())

# =========================
# 2. Gráfica número de casos por semana
# =========================

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df["ds"],
        y=df["y"],
        mode='lines+markers',
        name='Casos de Dengue',
        line=dict(color='#2196F3', width=1.8),
        marker=dict(size=4)
    )
)

fig.update_layout(
    title='Número de Casos de Dengue por Semana',
    xaxis_title='Semana Epidemiológica',
    yaxis_title='Número de Casos',
    height=450,
    template='plotly_white'
)

fig.show()

### 🔍 ¿Qué observar?
- **Estacionalidad anual marcada:** cada diciembre hay un pico pronunciado (demanda navideña)
- **Tendencia leve:** el nivel promedio crece ligeramente a lo largo del período
- **Heteroscedasticidad:** las fluctuaciones estacionales parecen crecer con el nivel → candidato para log-transformación


---
## 🔍 PARTE 3: Calidad de Datos

Antes de cualquier análisis o modelado, hay que garantizar la calidad de los datos.
Problemas sin detectar aquí se propagan a todos los modelos y métricas.

Trabajamos con tres tipos de problemas:

| Problema | Síntoma | Estrategia |
|----------|---------|-----------|
| **Valores faltantes** | NaN o huecos temporales | Forward fill, interpolación, media estacional |
| **Outliers** | Puntos extremos atípicos | IQR, Z-score, Hampel |
| **Cambios de régimen** | Saltos estructurales abruptos | Inspección visual, prueba de Chow |


### 3.1 Valores Faltantes — Detección y Estrategias de Imputación

In [6]:
import pandas as pd

# Convert 'ds' column to datetime objects to allow date range operations
#df['ds'] = pd.to_datetime(df['ds'].astype(str) + '-1', format='%Y-W%W-%w')

# ── Detectar valores faltantes ──────────────────────────────────────────
print("=" * 55)
print("  DIAGNÓSTICO DE VALORES FALTANTES")
print("=" * 55)

# Verificar continuidad temporal
expected_dates = pd.date_range(df.ds.min(), df.ds.max(), freq='W-MON')
missing_dates  = expected_dates.difference(df.ds)

print(f"  Observaciones esperadas: {len(expected_dates)}")
print(f"  Observaciones presentes: {len(df)}")
print(f"  Fechas faltantes:        {len(missing_dates)}")
print(f"  NaN en columna y:        {df.y.isna().sum()}")

if len(missing_dates) == 0 and df.y.isna().sum() == 0:
    print("\n  ✅ Serie completa — sin valores faltantes")
else:
    print(f"\n  ⚠️  Fechas faltantes: {missing_dates.tolist()}")

  DIAGNÓSTICO DE VALORES FALTANTES
  Observaciones esperadas: 744
  Observaciones presentes: 741
  Fechas faltantes:        3
  NaN en columna y:        0

  ⚠️  Fechas faltantes: [Timestamp('2024-02-26 00:00:00'), Timestamp('2024-03-11 00:00:00'), Timestamp('2024-03-18 00:00:00')]


### Imputación de Valores Faltantes por Media Estacional

Dado que la serie presenta una estacionalidad marcada, la estrategia más adecuada es la imputación por la media estacional. Esta técnica sustituye los valores faltantes por el promedio de los valores observados para el mismo período (e.g., misma semana o mes) a lo largo de la serie histórica.

In [7]:
# 1. Crear un DataFrame completo con todas las fechas esperadas
full_date_range = pd.date_range(df.ds.min(), df.ds.max(), freq='W-MON')
full_df = pd.DataFrame({'ds': full_date_range, 'unique_id': 'dengue_cali'})

# 2. Merge con el DataFrame original para identificar los huecos
df_merged = pd.merge(full_df, df, on=['ds', 'unique_id'], how='left')

# 3. Calcular la media estacional (por semana del año)
df_merged['week_of_year'] = df_merged['ds'].dt.isocalendar().week
seasonal_means = df_merged.groupby('week_of_year')['y'].mean().to_dict()

# 4. Imputar los valores faltantes usando la media estacional
def impute_seasonal(row):
    if pd.isna(row['y']):
        return seasonal_means.get(row['week_of_year'], row['y']) # Use .get to handle weeks not in seasonal_means
    return row['y']

df_imputed = df_merged.copy()
df_imputed['y'] = df_imputed.apply(impute_seasonal, axis=1)

# 5. Verificar que no queden NaN en 'y'
print(f"NaN después de imputación: {df_imputed['y'].isna().sum()}")

# 6. Eliminar la columna temporal 'week_of_year'
df_imputed = df_imputed.drop(columns=['week_of_year'])

# Actualizar el DataFrame original 'df' con los datos imputados
df = df_imputed.copy()

print("DataFrame 'df' actualizado con los valores imputados.")
print(df.head())
print(df.tail())

NaN después de imputación: 0
DataFrame 'df' actualizado con los valores imputados.
          ds    unique_id      y
0 2009-12-28  dengue_cali   12.0
1 2010-01-04  dengue_cali  142.0
2 2010-01-11  dengue_cali  210.0
3 2010-01-18  dengue_cali  253.0
4 2010-01-25  dengue_cali  345.0
            ds    unique_id           y
739 2024-02-26  dengue_cali  132.071429
740 2024-03-04  dengue_cali    5.000000
741 2024-03-11  dengue_cali  113.428571
742 2024-03-18  dengue_cali   97.428571
743 2024-03-25  dengue_cali    1.000000


### 3.2 Detección de Outliers — IQR y Z-score

In [8]:
# ── Método 1: IQR (Rango Intercuartílico) ────────────────────────────────
Q1, Q3 = df.y.quantile(0.25), df.y.quantile(0.75)
IQR    = Q3 - Q1
limite_inf_iqr = Q1 - 1.5 * IQR
limite_sup_iqr = Q3 + 1.5 * IQR
outliers_iqr = df[(df.y < limite_inf_iqr) | (df.y > limite_sup_iqr)]

print("IQR — Detección de Outliers")
print(f"  Q1={Q1:.0f}  Q3={Q3:.0f}  IQR={IQR:.0f}")
print(f"  Límite inferior: {limite_inf_iqr:.0f}")
print(f"  Límite superior: {limite_sup_iqr:.0f}")
print(f"  Outliers detectados: {len(outliers_iqr)}")
print(outliers_iqr[['ds','y']])

# ── Método 2: Z-score ─────────────────────────────────────────────────────
z_scores = np.abs(stats.zscore(df.y))
outliers_z = df[z_scores > 3]
print(f"\nZ-score (|z| > 3) — Outliers detectados: {len(outliers_z)}")
if len(outliers_z): print(outliers_z[['ds','y']])

# ── Visualización comparativa ─────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.ds, y=df.y, mode='lines',
                          name='Serie', line=dict(color='#2196F3', width=1.5)))
fig.add_hline(y=limite_sup_iqr, line_dash='dash', line_color='orange',
              annotation_text='Límite IQR superior')
fig.add_hline(y=limite_inf_iqr, line_dash='dash', line_color='orange',
              annotation_text='Límite IQR inferior')
if len(outliers_iqr):
    fig.add_trace(go.Scatter(x=outliers_iqr.ds, y=outliers_iqr.y,
                              mode='markers', name='Outlier (IQR)',
                              marker=dict(color='red', size=10, symbol='circle-open', line_width=2)))
fig.update_layout(title='Detección de Outliers — Método IQR',
                  height=380, template='plotly_white',
                  xaxis_title='Fecha', yaxis_title='Ventas')
fig.show()

IQR — Detección de Outliers
  Q1=17  Q3=107  IQR=90
  Límite inferior: -118
  Límite superior: 242
  Outliers detectados: 57
            ds      y
3   2010-01-18  253.0
4   2010-01-25  345.0
5   2010-02-01  375.0
6   2010-02-08  504.0
7   2010-02-15  573.0
8   2010-02-22  543.0
9   2010-03-01  475.0
10  2010-03-08  451.0
11  2010-03-15  439.0
12  2010-03-22  384.0
13  2010-03-29  304.0
14  2010-04-05  330.0
15  2010-04-12  308.0
16  2010-04-19  293.0
172 2013-04-15  255.0
316 2016-01-18  281.0
317 2016-01-25  272.0
318 2016-02-01  310.0
319 2016-02-08  365.0
320 2016-02-15  290.0
321 2016-02-22  351.0
322 2016-02-29  335.0
323 2016-03-07  331.0
324 2016-03-14  247.0
327 2016-04-04  255.0
524 2020-01-13  264.0
526 2020-01-27  438.0
527 2020-02-03  348.0
528 2020-02-10  338.0
529 2020-02-17  359.0
530 2020-02-24  379.0
531 2020-03-02  305.0
532 2020-03-09  396.0
533 2020-03-16  271.0
534 2020-03-23  255.0
708 2023-07-24  273.0
710 2023-08-07  264.0
711 2023-08-14  294.0
712 2023-08-21  2

### 🔍 Interpretación — Outliers
## Interpretación de los Outliers — Casos de Dengue

Los outliers detectados por el método IQR no parecen ser errores de datos, sino eventos epidemiológicos reales de alta importancia.

# 1. No son ruido, son brotes epidémicos

Los puntos rojos aparecen concentrados en los picos de la serie:

* 2010
* 2013
* 2016
* 2020
* 2023 - 2024

Esto indica que representan:

## Semanas de transmisión excepcionalmente alta

es decir:

### brotes epidémicos severos

y no simples anomalías estadísticas.

En dengue esto es completamente esperado.

# 2. Reflejan cambios estructurales del sistema

Estos outliers suelen estar asociados a:

* Cambios climáticos extremos
* Fenómeno de El Niño / La Niña
* Aumento de lluvias
* Aumento de temperatura
* Aparición de nuevos serotipos
* Disminución de inmunidad poblacional
* Fallas en campañas de control vectorial
* Saturación hospitalaria
* Subregistro previo seguido de corrección

Es decir: representan cambios reales del proceso epidemiológico, no errores de medición.

# 3. Alta concentración en ciertos años
Especialmente fuerte en:
2010 y 2023–2024

Esto sugiere que esos años fueron años epidémicos extraordinarios y probablemente deberían analizarse por separado.

Pueden incluso justificar:

* Segmentación del modelo
* Análisis por régimen epidemiológico
* Modelos con change points

# 4. Importancia para modelado predictivo

Eliminar estos outliers sería un error.

### modelarlos correctamente

con:

* Negative Binomial
* Poisson
* LSTM robusto
* modelos híbridos
* Regime Switching Models



In [9]:
df_check = df.copy()
# Convert 'ds' column to datetime objects
# df_check['ds'] = pd.to_datetime(df_check['ds'].astype(str) + '-1', format='%Y-W%W-%w') # This line is no longer needed as 'ds' is already datetime
df_check['mes'] = df_check.ds.dt.month
df_check['outlier_mensual'] = False

for mes in range(1, 13):
    mask = df_check.mes == mes
    vals_mes = df_check.loc[mask, 'y']
    q1, q3 = vals_mes.quantile(0.25), vals_mes.quantile(0.75)
    iqr_m   = q3 - q1
    es_outlier = (vals_mes < q1 - 1.5*iqr_m) | (vals_mes > q3 + 1.5*iqr_m)
    df_check.loc[mask & es_outlier, 'outlier_mensual'] = True

n_out_mes = df_check.outlier_mensual.sum()
print(f"Outliers con IQR por mes: {n_out_mes}")
if n_out_mes:
    print(df_check[df_check.outlier_mensual][['ds','y','mes']])
else:
    print("✅ Sin outliers cuando se aplica IQR por estación")

Outliers con IQR por mes: 56
            ds      y  mes
3   2010-01-18  253.0    1
4   2010-01-25  345.0    1
5   2010-02-01  375.0    2
6   2010-02-08  504.0    2
7   2010-02-15  573.0    2
8   2010-02-22  543.0    2
9   2010-03-01  475.0    3
10  2010-03-08  451.0    3
11  2010-03-15  439.0    3
12  2010-03-22  384.0    3
184 2013-07-08  238.0    7
185 2013-07-15  202.0    7
186 2013-07-22  213.0    7
189 2013-08-12  177.0    8
197 2013-10-07  147.0   10
198 2013-10-14  165.0   10
199 2013-10-21  183.0   10
200 2013-10-28  161.0   10
316 2016-01-18  281.0    1
317 2016-01-25  272.0    1
319 2016-02-08  365.0    2
321 2016-02-22  351.0    2
322 2016-02-29  335.0    2
323 2016-03-07  331.0    3
524 2020-01-13  264.0    1
526 2020-01-27  438.0    1
527 2020-02-03  348.0    2
528 2020-02-10  338.0    2
529 2020-02-17  359.0    2
530 2020-02-24  379.0    2
532 2020-03-09  396.0    3
706 2023-07-10  211.0    7
707 2023-07-17  235.0    7
708 2023-07-24  273.0    7
709 2023-07-31  221.0    7

### 3.3 Cambios de Régimen y Anomalías Estructurales

In [10]:
# ── Detección visual: media y varianza móviles ──────────────────────────
ventana = 2
media_movil  = df.y.rolling(ventana, center=True).mean()
std_movil    = df.y.rolling(ventana, center=True).std()

fig = make_subplots(rows=2, cols=1,
    subplot_titles=['Serie + Media Móvil (ventana=2)',
                    'Desv. Estándar Móvil — cambios indican heteroscedasticidad'],
    vertical_spacing=0.15)

fig.add_trace(go.Scatter(x=df.ds, y=df.y, mode='lines', name='Original',
    line=dict(color='lightblue', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.ds, y=media_movil, mode='lines', name='Media móvil',
    line=dict(color='#2196F3', width=2.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.ds, y=std_movil, mode='lines', name='Desv. estándar',
    line=dict(color='#FF5722', width=2), fill='tozeroy',
    fillcolor='rgba(255,87,34,0.1)'), row=2, col=1)

fig.update_layout(height=550, template='plotly_white',
                  title='Detección de Cambios de Régimen — Estadísticas Móviles')
fig.show()

##Interpretación

Se observan claramente varios cambios de régimen:

###Régimen de alta transmisión

Cuando la media móvil sube fuertemente:

* 2010
* 2013 - 2014
* 2016
* 2020
* 2023 - 2024

Estos corresponden a:

Brotes epidémicos
Régimen de baja transmisión

Cuando la media móvil permanece cerca de cero:

* 2011 - 2012
* 2017 - 2019

Esto representa: Periodos endémicos controlados con baja circulación viral

In [11]:
# ── Prueba de Chow (concepto): comparar dos subperíodos ─────────────────
# Dividimos la serie en dos mitades y comparamos sus medias y varianzas
# Una diferencia estadísticamente significativa sugiere cambio de régimen

# Ensure 'ds' column is datetime type for strftime to work
# This assumes 'ds' is in 'YYYY-Www' format; adding '-1' to parse it as Monday of that week.
# The 'ds' column is already datetime, so this line is no longer needed.
# df['ds'] = pd.to_datetime(df['ds'].astype(str) + '-1', format='%Y-W%W-%w')

n_total  = len(df)
mitad    = n_total // 2
y1 = df.y.values[:mitad]
y2 = df.y.values[mitad:]

# Test de Levene: igualdad de varianzas
stat_var, p_var = stats.levene(y1, y2)

# Test t: igualdad de medias
stat_med, p_med = stats.ttest_ind(y1, y2)

print("  TEST DE CAMBIO DE RÉGIMEN — Comparación de dos subperíodos")
print(f"  Período 1: {df.ds.iloc[0].strftime('%Y-%m')} → {df.ds.iloc[mitad-1].strftime('%Y-%m')}")
print(f"  Período 2: {df.ds.iloc[mitad].strftime('%Y-%m')} → {df.ds.iloc[-1].strftime('%Y-%m')}")
print()
print(f"  Media P1={y1.mean():.0f}  |  Media P2={y2.mean():.0f}")
print(f"  Desv. P1={y1.std():.0f}   |  Desv. P2={y2.std():.0f}")
print()
print(f"  Test Levene (varianzas iguales): stat={stat_var:.3f}  p={p_var:.4f}",
      "→ Varianzas distintas ⚠️" if p_var < 0.05 else "→ Varianzas similares ✅")
print(f"  Test t (medias iguales):         stat={stat_med:.3f}  p={p_med:.4f}",
      "→ Medias distintas ⚠️" if p_med < 0.05 else "→ Medias similares ✅")

print()
print("  Interpretación:")
if p_var < 0.05 or p_med < 0.05:
    print("  ⚠️  Hay evidencia de cambio estructural entre períodos.")
    print("     Estrategias: dummy variable, segmentar la serie, o modelar con SARIMA.")
else:
    print("  ✅  Los dos subperíodos son estadísticamente similares.")
    print("     La serie no muestra cambios de régimen significativos.")


  TEST DE CAMBIO DE RÉGIMEN — Comparación de dos subperíodos
  Período 1: 2009-12 → 2017-02
  Período 2: 2017-02 → 2024-03

  Media P1=96  |  Media P2=72
  Desv. P1=96   |  Desv. P2=97

  Test Levene (varianzas iguales): stat=2.803  p=0.0945 → Varianzas similares ✅
  Test t (medias iguales):         stat=3.436  p=0.0006 → Medias distintas ⚠️

  Interpretación:
  ⚠️  Hay evidencia de cambio estructural entre períodos.
     Estrategias: dummy variable, segmentar la serie, o modelar con SARIMA.


### 🔍 Interpretación — Cambios de Régimen

Un **cambio de régimen** ocurre cuando las propiedades estadísticas de la serie cambian abruptamente
(por ejemplo, tras un evento económico, una regulación o un shock externo).

**Estrategias de tratamiento:**

| Estrategia | Cuándo usar |
|-----------|-------------|
| **Variable dummy** | Cambio bien localizado en el tiempo |
| **Segmentar la serie** | Cambio permanente — modelar cada período por separado |
| **Modelo con cambio de punto** | Detección automática (e.g., Prophet, BOCPD) |
| **Remover la tendencia** | Si el "cambio" es en realidad una tendencia no capturada |

> 📌 En esta sesión el diagnóstico es **visual y estadístico básico**. Técnicas más avanzadas
> de detección automática de puntos de cambio (BOCPD, PELT) se verán en semanas posteriores.


---
## 📈 PARTE 4: Análisis de Autocorrelación (ACF)

La ACF mide cuánto se parece la serie a sí misma en distintos instantes del pasado.
Aquí la usamos para confirmar que la serie tiene estructura temporal predecible.

> 📌 Recordatorio de la **Sesión 2:** la ACF de una serie estacionaria decae rápidamente;
> la de una serie no estacionaria decae lentamente. La ACF que veremos aquí también revelará
> el período estacional de la serie.


In [12]:
def plot_acf_plotly(series, title, n_lags=30, color='#2196F3'):
    """
    ACF interactivo con Plotly.
    Barras en rojo = significativas al 95% (fuera del IC).
    """
    arr = np.asarray(series).astype(float)
    arr = arr[~np.isnan(arr)]
    acf_vals = acf(arr, nlags=n_lags, fft=True)
    ci   = 1.96 / np.sqrt(len(arr))
    lags = np.arange(len(acf_vals))

    fig = go.Figure()
    for lag in lags:
        bar_color = color if abs(acf_vals[lag]) <= ci else 'crimson'
        fig.add_trace(go.Scatter(x=[lag, lag], y=[0, acf_vals[lag]], mode='lines',
                                  line=dict(color=bar_color, width=2.5), showlegend=False))
    fig.add_trace(go.Scatter(x=lags, y=acf_vals, mode='markers', showlegend=False,
                              marker=dict(color=[color if abs(v) <= ci else 'crimson'
                                                 for v in acf_vals], size=6)))
    fig.add_hline(y= ci, line_dash='dash', line_color='gray', opacity=0.7,
                  annotation_text=f'IC 95% = ±{ci:.3f}')
    fig.add_hline(y=-ci, line_dash='dash', line_color='gray', opacity=0.7)
    fig.add_hline(y=0,   line_color='black', line_width=0.8)
    fig.update_layout(title=title, xaxis_title='Lag', yaxis_title='Autocorrelación',
                      height=360, template='plotly_white', yaxis=dict(range=[-1.05, 1.05]))
    return fig

fig_acf = plot_acf_plotly(df.y, 'ACF - Casos de Dengue en la ciudad de Cali', n_lags=30)
fig_acf.show()

##1. Autocorrelación muy alta

Se observa que los primeros lags tienen valores muy cercanos a 1.
Esto significa que los casos actuales dependen fuertemente del pasado reciente

Es decir si esta semana hay muchos casos,la siguiente semana probablemente también.

Esto es completamente esperado en dengue.

2. Decaimiento lento La autocorrelación disminuye lentamente no cae bruscamente permanece significativa hasta lag 25+

Esto indica una fuerte persistencia temporal y también posible no estacionariedad, porque en series estacionarias la ACF suele caer mucho más rápido.

Aquí no ocurre eso.

3. No parece ruido blanco Si fuera ruido blanco:

ACF(k)≈0 para casi todos los lags.

Pero aquí casi todos los lags son significativos por encima del IC 95%, entonces la serie tiene estructura temporal real no es aleatoria.

4. Posible estacionalidad

Como la ACF se mantiene alta durante muchos rezagos puede existir estacionalidad de mediano plazo (posiblemente anual en semanas epidemiológicas)

Por ejemplo: ciclos climáticos, lluvias, temperatura, comportamiento vectorial

Esto suele confirmarse mejor con:

PACF
STL decomposition
Fourier terms
5. Implicación para modelado

Esto sugiere que usar información pasada es obligatorio, por eso funcionan bien:

ARIMA
SARIMA
LSTM
Modelos híbridos

Porque todos aprovechan memoria temporal.

Lo más importante La ACF confirma que el sistema tiene memoria epidemiológica y no es un proceso instantáneo.

Hay dependencia entre semanas, lo cual es exactamente consistente con modelos SIR-SI, porque infectados actuales generan infectados futuros


---
## 📐 PARTE 4.5: Tests de Estacionaridad — ADF y KPSS

La ACF mostró decaimiento lento → la serie NO es estacionaria. Los tests **ADF y KPSS**
formalizan esta observación con evidencia estadística.

> ### ¿Por qué hacer esto antes del modelado baseline?
>
> El diagnóstico de estacionaridad forma parte del **análisis exploratorio (EDA)**,
> no del preprocesamiento previo al modelo. La razón:
>
> | Tipo de modelo | ¿Requiere transformar para estacionaridad? | ¿Cómo maneja la no estacionaridad? |
> |---------------|---------------------------------------------|-------------------------------------|
> | **Naive / SeasonalNaive** | ❌ No | La copia mecánicamente |
> | **WindowAverage** | ❌ No | Promedia los valores crudos |
> | **RandomWalkWithDrift** | ❌ No | Fue diseñado para series con drift |
> | **ARIMA / SARIMA** | ✅ Sí | La elimina vía diferenciación (parámetro `d`) |
> | **ML (XGBoost, LightGBM)** | ⚡ Opcional | Features de lag la capturan implícitamente |
> | **DL (RNN, Transformer)** | ⚡ Opcional | Aprenden el patrón directamente |
>
> **Conclusión práctica:** los baselines funcionan sobre datos crudos. Sin embargo,
> el diagnóstico de estacionaridad aquí nos permite:
> - Confirmar qué tipo de serie estamos enfrentando
> - Entender por qué SeasonalNaive funcionará bien y Naive no
> - Preparar el terreno para ARIMA en la próxima sesión, donde sí será obligatorio


In [13]:
# ── Tests ADF y KPSS sobre la serie de casos de Dengue ─────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss

def test_estacionaridad(serie, nombre):
    """
    Aplica ADF + KPSS y entrega la conclusión conjunta.
    Tabla de decisión:
      ADF p<0.05  + KPSS p≥0.05  → ESTACIONARIA
      ADF p≥0.05  + KPSS p<0.05  → NO ESTACIONARIA
      Ambos rechazan / ambos aceptan → INCIERTA
    """
    arr = np.asarray(serie).astype(float)
    arr = arr[~np.isnan(arr)]

    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(arr, autolag='AIC')
    kpss_stat, kpss_p, _, kpss_crit    = kpss(arr, regression='c', nlags='auto')

    adf_est  = adf_p  < 0.05
    kpss_est = kpss_p >= 0.05

    if   adf_est and kpss_est:      conclusion = "✅  ESTACIONARIA"
    elif not adf_est and not kpss_est: conclusion = "❌  NO ESTACIONARIA"
    elif adf_est and not kpss_est:   conclusion = "⚠️  INCIERTA (tendencia?)"
    else:                             conclusion = "⚠️  INCIERTA (cerca del límite)"

    print(f"\n{'═'*58}")
    print(f"  {nombre}")
    print(f"{'═'*58}")
    print(f"  ADF:  stat={adf_stat:8.4f}  p={adf_p:.4f}",
          "→ ES estacionaria ✅" if adf_est else "→ NO estacionaria ❌")
    print(f"  KPSS: stat={kpss_stat:8.4f}  p={kpss_p:.4f}",
          "→ ES estacionaria ✅" if kpss_est else "→ NO estacionaria ❌")
    print(f"  {'─'*54}")
    print(f"  CONCLUSIÓN: {conclusion}")
    return {'adf_p': adf_p, 'kpss_p': kpss_p, 'estacionaria': adf_est and kpss_est}

# Aplicar a la serie original
r_orig = test_estacionaridad(df.y, 'Casos de dengue en Cali — Serie Original')


══════════════════════════════════════════════════════════
  Casos de dengue en Cali — Serie Original
══════════════════════════════════════════════════════════
  ADF:  stat= -5.0959  p=0.0000 → ES estacionaria ✅
  KPSS: stat=  0.1362  p=0.1000 → ES estacionaria ✅
  ──────────────────────────────────────────────────────
  CONCLUSIÓN: ✅  ESTACIONARIA


### 🔍 Interpretación — Tests de Estacionaridad

El código imprimirá el resultado automático. La interpretación general:

- **ADF p > 0.05** → No rechaza raíz unitaria → evidencia de NO estacionaridad ❌
- **KPSS p ≥ 0.05** → No rechaza estacionaridad → evidencia de estacionaridad ✅ *(contradictorio)*
- **Resultado frecuente en esta serie:** caso *incierto* o *cerca del límite*

> 📌 Este es un resultado pedagógicamente valioso:
> **Los tests no siempre coinciden.** La ACF con decaimiento lento y los picos estacionales
> son la evidencia visual más robusta de que la serie tiene estructura no estacionaria.
>
> **Regla práctica:** cuando hay resultados contradictorios, confiar en la evidencia visual
> (ACF, media/std móvil) y diferenciar conservadoramente antes de modelar con ARIMA.
> Para baselines, esto **no cambia nada** — operan sobre los datos crudos.


---
## 🔬 PARTE 5: Test de Ljung-Box

El **Test de Ljung-Box** prueba si múltiples autocorrelaciones son simultáneamente cero.
Nos dice si la serie tiene estructura temporal explotable por un modelo.

$$H_0: \rho_1 = \rho_2 = \cdots = \rho_h = 0 \quad \Rightarrow \quad Q_{LB} = n(n+2)\sum_{k=1}^{h}\frac{\hat{\rho}_k^2}{n-k} \sim \chi^2_h$$


In [14]:
# ── Test de Ljung-Box para múltiples lags ────────────────────────────────
lags_test = [1, 6, 12, 18, 24]

print("TEST DE LJUNG-BOX — Casos de Dengue Cali")
print("H₀: ρ₁ = ρ₂ = ··· = ρₕ = 0  (no hay autocorrelación)")
print("─" * 62)
print(f"{'Lags':>6}  {'Estadístico Q':>14}  {'p-valor':>10}  {'Conclusión'}")
print("─" * 62)

for lag in lags_test:
    result = acorr_ljungbox(df.y.values, lags=[lag], return_df=True)
    q_stat = result['lb_stat'].iloc[0]
    p_val  = result['lb_pvalue'].iloc[0]
    conc   = "Rechaza H₀ — HAY autocorrelación ❌" if p_val < 0.05              else "No rechaza H₀ — sin autocorrelación ✅"
    print(f"  {lag:>4}  {q_stat:>14.4f}  {p_val:>10.6f}  {conc}")
print("─" * 62)

TEST DE LJUNG-BOX — Casos de Dengue Cali
H₀: ρ₁ = ρ₂ = ··· = ρₕ = 0  (no hay autocorrelación)
──────────────────────────────────────────────────────────────
  Lags   Estadístico Q     p-valor  Conclusión
──────────────────────────────────────────────────────────────
     1        679.4849    0.000000  Rechaza H₀ — HAY autocorrelación ❌
     6       3260.3370    0.000000  Rechaza H₀ — HAY autocorrelación ❌
    12       4850.0984    0.000000  Rechaza H₀ — HAY autocorrelación ❌
    18       5565.2352    0.000000  Rechaza H₀ — HAY autocorrelación ❌
    24       5782.7130    0.000000  Rechaza H₀ — HAY autocorrelación ❌
──────────────────────────────────────────────────────────────


In [15]:
# ── Visualización del p-valor por lag ────────────────────────────────────
lags_all   = list(range(1, 31))
results_lb = acorr_ljungbox(df.y.values, lags=lags_all, return_df=True)
p_valores  = results_lb['lb_pvalue'].values

fig = go.Figure()
fig.add_trace(go.Scatter(x=lags_all, y=p_valores, mode='lines+markers',
                          line=dict(color='#2196F3', width=2),
                          marker=dict(color=['red' if p < 0.05 else 'green'
                                             for p in p_valores], size=7),
                          name='p-valor Ljung-Box'))
fig.add_hline(y=0.05, line_dash='dash', line_color='red',
              annotation_text='α = 0.05', annotation_position='right')

fig.update_layout(
    title='Test de Ljung-Box — p-valores por lag<br>'
          '<sup>Puntos rojos = evidencia de autocorrelación significativa</sup>',
    xaxis_title='Lag', yaxis_title='p-valor',
    height=360, template='plotly_white', yaxis=dict(range=[-0.05, 1.05])
)
fig.show()

### 🔍 Interpretación — Ljung-Box

## Interpretación del Test de Ljung-Box

Esta gráfica evalúa si la serie temporal tiene autocorrelación significativa o si se comporta como ruido blanco.

# ¿Qué prueba Ljung-Box?

La hipótesis nula es:

es decir:

## no existe autocorrelación

(la serie sería aleatoria / ruido blanco)

La hipótesis alternativa:

## sí existe autocorrelación

# Qué muestra la gráfica

* eje X → lag (rezago)
* eje Y → p-value

La línea roja punteada representa:

## nivel de significancia

\alpha = 0.05

# Interpretación principal

Se observa que todos los p-valores están por debajo de 0.05

Esto significa que se rechaza H₀ en todos los lags, por lo tanto sí existe autocorrelación significativa y además muy fuerte.

La serie NO es ruido blanco

Tiene:

* Memoria temporal
* Dependencia entre semanas
* Estructura epidemiológica real

Es decir:

si hoy hay muchos casos, las siguientes semanas estarán influenciadas por eso.

Muy típico en enfermedades infecciosas.

# Relación con tu ACF

Esto confirma exactamente lo que mostró la ACF:

* Alta persistencia
* Decaimiento lento
* Dependencia temporal fuerte

La ACF lo muestra visualmente.

Ljung-Box lo confirma estadísticamente.

# Interpretación epidemiológica

Esto significa que el dengue tienepropagación dependiente del tiempo

porque:

* Infectados actuales generan nuevos infectados
* El mosquito mantiene la transmisión
* El clima mantiene condiciones favorables

No es un evento aislado.

Es un proceso dinámico.

# Implicación para modelado

No debes usar modelos que asuman independencia.

Sí debes usar:

* ARIMA / SARIMA
* LSTM
* modelos híbridos
* modelos SIR-SI

porque aprovechan dependencia temporal.

---

# Conclusión fuerte

La gráfica demuestra:

### ✔ autocorrelación significativa

### ✔ no es ruido blanco

### ✔ fuerte dependencia temporal

### ✔ evidencia de no estacionariedad

### ✔ alta memoria epidemiológica



---
## ✂️ PARTE 6: Train-Test Split Temporal

**Regla crítica:** NUNCA aleatorizar. El tiempo tiene dirección y los modelos de series
temporales aprenden del pasado para predecir el futuro.


In [52]:
# ── Split temporal 70/30 ────────────────────────────────────────────────
# Change fecha_corte to a date relevant for the dengue dataset (starts 2010)
fecha_corte = '2020-01-01'
train = df[df.ds < fecha_corte].copy()
test  = df[df.ds >= fecha_corte].copy()

print(f"TRAIN: {len(train)} observaciones  ({train.ds.min().strftime('%Y-%m')} → {train.ds.max().strftime('%Y-%m')})")
print(f"TEST:  {len(test)}  observaciones  ({test.ds.min().strftime('%Y-%m')} → {test.ds.max().strftime('%Y-%m')})")
print(f"Proporción: {len(train)/len(df)*100:.0f}% / {len(test)/len(df)*100:.0f}%")

# Visualización del split
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.ds, y=train.y, mode='lines+markers',
                          name='Train', line=dict(color='#2196F3', width=2),
                          marker=dict(size=3)))
fig.add_trace(go.Scatter(x=test.ds, y=test.y, mode='lines+markers',
                          name='Test', line=dict(color='#FF5722', width=2),
                          marker=dict(size=3)))
# add_shape evita el bug de pandas 2.0+: add_vline falla con fechas
fig.add_shape(type='line', xref='x', yref='paper',
              x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
              line=dict(dash='dash', color='gray', width=1.5))
fig.add_annotation(x=fecha_corte, y=1.02, yref='paper',
                   text='Corte', showarrow=False,
                   font=dict(size=10, color='gray'), xanchor='left')
fig.update_layout(title=f'Train-Test Split Temporal — {len(train)/len(df)*100:.0f}% / {len(test)/len(df)*100:.0f}%',
                  xaxis_title='Fecha', yaxis_title='Casos',
                  height=380, template='plotly_white')
fig.show()

TRAIN: 523 observaciones  (2009-12 → 2019-12)
TEST:  221  observaciones  (2020-01 → 2024-03)
Proporción: 70% / 30%


In [55]:
FECHA_CORTE = "2020-01-01"

def dividir_train_test_temporal(
    df,
    fecha_corte="2020-01-01",
    columna_fecha="ds"
):
    df = df.copy()
    df[columna_fecha] = pd.to_datetime(df[columna_fecha])
    df = df.sort_values(columna_fecha)

    train = df[df[columna_fecha] < fecha_corte].copy()
    test = df[df[columna_fecha] >= fecha_corte].copy()

    resumen = pd.DataFrame({
        "Segmento": ["Entrenamiento", "Prueba"],
        "Observaciones": [len(train), len(test)],
        "Inicio": [train[columna_fecha].min(), test[columna_fecha].min()],
        "Fin": [train[columna_fecha].max(), test[columna_fecha].max()]
    })

    return train, test, resumen


train, test, resumen_particion = dividir_train_test_temporal(
    df,
    fecha_corte=FECHA_CORTE
)

display(resumen_particion)

print(f"TRAIN: {len(train)} observaciones")
print(f"TEST: {len(test)} observaciones")
print(f"Rango train: {train['ds'].min().date()} → {train['ds'].max().date()}")
print(f"Rango test:  {test['ds'].min().date()} → {test['ds'].max().date()}")

,Segmento,Observaciones,Inicio,Fin
0,Entrenamiento,523,2009-12-28,2019-12-30
1,Prueba,221,2020-01-06,2024-03-25


TRAIN: 523 observaciones
TEST: 221 observaciones
Rango train: 2009-12-28 → 2019-12-30
Rango test:  2020-01-06 → 2024-03-25


In [53]:
# Visualización del split
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.ds, y=train.y, mode='lines+markers',
                          name='Train', line=dict(color='#2196F3', width=2),
                          marker=dict(size=3)))
fig.add_trace(go.Scatter(x=test.ds, y=test.y, mode='lines+markers',
                          name='Test', line=dict(color='#FF5722', width=2),
                          marker=dict(size=3)))
# add_shape evita el bug de pandas 2.0+: add_vline falla con fechas
fig.add_shape(type='line', xref='x', yref='paper',
              x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
              line=dict(dash='dash', color='gray', width=1.5))
fig.add_annotation(x=fecha_corte, y=1.02, yref='paper',
                   text='Corte', showarrow=False,
                   font=dict(size=10, color='gray'), xanchor='left')
fig.update_layout(title=f'Train-Test Split Temporal — {len(train)/len(df)*100:.0f}% / {len(test)/len(df)*100:.0f}%',
                  xaxis_title='Fecha', yaxis_title='Casos',
                  height=380, template='plotly_white')
fig.show()

---
## 🤖 PARTE 7: Baselines con `statsforecast`

Implementamos los 4 modelos baseline vistos en clase usando `StatsForecast`.
El patrón es siempre el mismo: `fit(train)` → `predict(h=horizonte)`.

> Este es el **patrón estándar de Nixtla** que usaremos con ARIMA, ML y DL en las semanas siguientes.


In [56]:
# ── Instanciar y ajustar todos los modelos en una sola llamada ──────────
h = len(test)   # horizonte = longitud del conjunto de prueba

sf = StatsForecast(
    models=[
        Naive(),                        # ŷ = y_t (último valor)
        SeasonalNaive(season_length=208), # ŷ = y_{t-212} (misma semana año anterior)
        WindowAverage(window_size=4),    # ŷ = promedio últimas 4 obs.
        RandomWalkWithDrift()            # ŷ = y_t + drift (tendencia histórica)
    ],
    freq='W-MON'    # frecuencia semanal
)

sf.fit(train)
preds = sf.predict(h=h)

print("Pronósticos generados:")
print(f"  Modelos: {[c for c in preds.columns if c not in ['unique_id','ds']]}")
print(f"  Horizonte: {h} meses")
preds.head()

Pronósticos generados:
  Modelos: ['Naive', 'SeasonalNaive', 'WindowAverage', 'RWD']
  Horizonte: 221 meses


,unique_id,ds,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,2020-01-06,162.0,216.0,130.5,162.287356
1,dengue_cali,2020-01-13,162.0,281.0,130.5,162.574713
2,dengue_cali,2020-01-20,162.0,272.0,130.5,162.862069
3,dengue_cali,2020-01-27,162.0,310.0,130.5,163.149425
4,dengue_cali,2020-02-03,162.0,365.0,130.5,163.436782


In [90]:
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, WindowAverage, RandomWalkWithDrift

HORIZONTE = len(test)   # horizonte = longitud del conjunto de prueba
PERIODO_ESTACIONALIDAD = 52*4 # 52 semanas * 4 años = 208 semanas para capturar la estacionalidad anual
VENTANA_MEDIA_MOVIL = 4 # Promedio de las últimas 4 semanas para suavizar ruido y capturar tendencia reciente

modelos_baseline = [
    Naive(),
    SeasonalNaive(season_length=PERIODO_ESTACIONALIDAD),
    WindowAverage(window_size=VENTANA_MEDIA_MOVIL),
    RandomWalkWithDrift()
]

sf = StatsForecast(
    models=modelos_baseline,
    freq="W-MON",
    n_jobs=-1
)

forecast_baseline = sf.forecast(
    df=train[["unique_id", "ds", "y"]],
    h=HORIZONTE
)

forecast_baseline.head()
preds=forecast_baseline.copy()

In [91]:
# ── Visualización: pronósticos vs valores reales ─────────────────────────
test_preds = test.merge(preds, on=['unique_id', 'ds'])
modelos    = ['Naive', 'SeasonalNaive', 'WindowAverage', 'RWD']
colores    = ['#FF5722', '#4CAF50', '#9C27B0', '#FF9800']

fig = go.Figure()

# Datos históricos completos (fondo)
fig.add_trace(go.Scatter(x=df.ds, y=df.y, mode='lines', name='Histórico',
                          line=dict(color='lightgray', width=1)))

# Período de prueba real
fig.add_trace(go.Scatter(x=test.ds, y=test.y, mode='lines+markers',
                          name='Real (test)', line=dict(color='black', width=1),
                          marker=dict(size=2)))

# Pronósticos de cada modelo
for modelo, color in zip(modelos, colores):
    fig.add_trace(go.Scatter(x=test_preds.ds, y=test_preds[modelo],
                              mode='lines+markers', name=modelo,
                              line=dict(color=color, width=1, dash='dot'),
                              marker=dict(size=2)))

fig.add_shape(type='line', xref='x', yref='paper',
              x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
              line=dict(dash='dash', color='gray', width=1))
fig.add_annotation(x=fecha_corte, y=1.02, yref='paper',
                   text='Inicio test', showarrow=False,
                   font=dict(size=10, color='gray'), xanchor='left')
fig.update_layout(
    title='Comparación de Baselines — Casos Dengue 2021-2023',
    xaxis_title='Fecha', yaxis_title='Casos',
    height=480, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

### 🔍 ¿Qué observar en los pronósticos?
## Interpretación — Comparación de Baselines

# Qué representa cada línea

## Gris — Histórico

Datos anteriores al periodo de prueba.

Sirve como contexto.

---

## Negro — Real (test)

Los valores verdaderos observados.

Es lo que realmente ocurrió.

Esta es la referencia más importante.

---

## Líneas de colores — Pronósticos

Cada una representa un modelo baseline:

* Naive
* SeasonalNaive
* WindowAverage
* RWD

Son predicciones simples.

---

# Observación principal

## Ningún baseline logra capturar el brote de 2023

Esto es lo más importante. Mientras los modelos predicen aproximadamente entre 40–60 casos, la realidad sube hasta más de 500 casos, eso significa una enorme subestimación del brote epidémico.

# Qué significa esto

Los modelos simplesfuncionan bien en periodos estables pero fallan completamente en cambios de régimen como epidemias fuertes.

Esto confirma que el sistema es no lineal y no puede modelarse bien con reglas simples.
SeasonalNaive parece el mejor baseline

Porque:

* Sigue mejor la estacionalidad
* Responde mejor que Naive puro

pero aun así:

## Falla gravemente en el brote real

---

# Evidencia de cambio estructural

La explosión de 2023 muestra:

## ruptura del patrón histórico

Es decir:

el comportamiento previo ya no explica bien el presente.

Esto sugiere:

* Regime shift
* Structural break
* Nueva dinámica epidemiológica

---

# La caída abrupta en 2024

Ese descenso casi a cero puede indicar:

* Dato incompleto
* corte de reporte
* Retraso de notificación
* Final de ventana de observación

Debe revisarse porque podría no ser real.

Eso es importante metodológicamente.

---

# Conclusión técnica

La gráfica demuestra:

### ✔ Fuerte subestimación de epidemias

### ✔ Falla de modelos lineales simples

### ✔ Existencia de cambio de régimen

### ✔ Ruptura estructural en 2023

### ✔ Necesidad de modelos avanzados


---
## 📊 PARTE 8: Métricas de Evaluación

Cuantificamos el error de cada modelo con las métricas vistas en clase.

| Métrica | Fórmula conceptual | Característica |
|---------|-------------------|---------------|
| **MAE** | Media de `\|error\|` | Robusta, misma unidad que los datos |
| **RMSE** | Raíz de la media de `error²` | Penaliza errores grandes |
| **sMAPE** | Error porcentual simétrico | Comparable entre series distintas |


In [20]:
# ── Calcular métricas con utilsforecast ──────────────────────────────────
modelos_col = ['Naive', 'SeasonalNaive', 'WindowAverage', 'RWD']

# MASE requiere la serie de entrenamiento como denominador (MAE del Naive sobre train)
naive_train_mae = float(np.abs(np.diff(train.y.values)).mean())

evaluacion = evaluate(
    test_preds,
    metrics=[mae, rmse, smape],
    models=modelos_col,
    target_col='y'
)

# MASE = MAE_test / MAE_naive_train
fila_mae = evaluacion[evaluacion.metric == 'mae'].copy()
nueva_fila_mase = fila_mae.copy()
nueva_fila_mase['metric'] = 'mase'

for m in modelos_col:
    nueva_fila_mase[m] = nueva_fila_mase[m] / naive_train_mae

# Concatenar a la tabla original
evaluacion = pd.concat([evaluacion, nueva_fila_mase], ignore_index=True)

print(f"\nMAE del Naive sobre train (denominador MASE): {naive_train_mae:.2f}")

# Reformatear para mejor visualización
print("=" * 60)
print("  RESULTADOS POR MÉTRICA — MODELOS BASELINE")
print("=" * 60)
display(evaluacion)

print("\n📌 Mejor modelo por métrica:")
for metrica in ['mae', 'rmse', 'smape', 'mase']:
    fila = evaluacion[evaluacion.metric == metrica]
    mejor = fila[modelos_col].values.argmin()
    nombre = modelos_col[mejor]
    valor  = fila[modelos_col].values[0][mejor]
    print(f"   {metrica.upper():>6}: {nombre}  ({valor:.4f})")


MAE del Naive sobre train (denominador MASE): 12.59
  RESULTADOS POR MÉTRICA — MODELOS BASELINE


,unique_id,metric,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,mae,111.860052,83.484486,99.798212,130.067416
1,dengue_cali,rmse,123.088358,122.358129,115.583418,140.008616
2,dengue_cali,smape,0.455027,0.599231,0.427979,0.482369
3,dengue_cali,mase,8.884806,6.630995,7.926760,10.330979



📌 Mejor modelo por métrica:
      MAE: SeasonalNaive  (83.4845)
     RMSE: WindowAverage  (115.5834)
    SMAPE: WindowAverage  (0.4280)
     MASE: SeasonalNaive  (6.6310)


In [21]:
# ── Visualización comparativa de métricas ────────────────────────────────
fig = make_subplots(rows=1, cols=3, subplot_titles=['MAE', 'RMSE', 'sMAPE'])
colores_m = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

for col_idx, metrica in enumerate(['mae', 'rmse', 'smape'], start=1):
    fila = evaluacion[evaluacion.metric == metrica]
    vals = [float(fila[m].values[0]) for m in modelos_col]
    mejor_idx = int(np.argmin(vals))

    bar_colors = ['gold' if i == mejor_idx else colores_m[i]
                  for i in range(len(modelos_col))]
    fig.add_trace(
        go.Bar(x=modelos_col, y=vals, marker_color=bar_colors,
               name=metrica, showlegend=False,
               text=[f'{v:.4f}' if metrica == 'smape' else f'{v:.1f}'
                     for v in vals], textposition='outside'),
        row=1, col=col_idx
    )

fig.update_layout(height=400, template='plotly_white',
                  title='Comparación de Métricas por Modelo<br>'
                        '<sup>Barra dorada = mejor modelo en esa métrica</sup>')
fig.show()

### 🔍 Conclusión de métricas

El menor error en todas las métricas es WindowsAverage

---
## 🔬 Cierre del Ciclo: Ljung-Box sobre Residuos del Mejor Modelo

El test de Ljung-Box tiene **dos usos** (como vimos en los slides):
1. **Serie cruda:** confirma que hay estructura predecible → conviene modelar
2. **Residuos del modelo:** verifica si quedaron patrones sin capturar → ¿es ruido blanco?

Si los residuos pasan el test (p ≥ 0.05), el modelo capturó todo lo disponible.
Si no, queda estructura que otro modelo (como ARIMA) podría aprovechar.


In [22]:
# ── Residuos del SeasonalNaive (mejor modelo) ────────────────────────────
residuos     = test_preds['y'] - test_preds['WindowAverage']
residuos_arr = residuos.values

print(f"Residuos — media: {residuos_arr.mean():.2f}  |  std: {residuos_arr.std():.2f}")
print(f"Esperado en ruido blanco: media ≈ 0, desviación estándar constante")

# ── Ljung-Box sobre los residuos ─────────────────────────────────────────
print("\nLJUNG-BOX SOBRE RESIDUOS DEL WindowAverage:")
print("─" * 58)
for lag in [1, 6, 12]:
    result = acorr_ljungbox(residuos_arr, lags=[lag], return_df=True)
    p_val  = result['lb_pvalue'].iloc[0]
    conc = "Quedan patrones ⚠️  → margen de mejora" if p_val < 0.05  else "Residuos ≈ ruido blanco ✅"
    print(f"  Lag {lag:>2}: p={p_val:.4f}  →  {conc}")

# ── ACF de los residuos (usa el acf importado en la celda de setup) ──────
acf_resid = acf(residuos_arr, nlags=52, fft=True)
ci_r = 1.96 / np.sqrt(len(residuos_arr))

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Residuos en el tiempo', 'ACF de Residuos'])

fig.add_trace(go.Scatter(x=test_preds.ds, y=residuos, mode='lines+markers',
    line=dict(color='#FF5722', width=2), name='Residuos'), row=1, col=1)
fig.add_hline(y=0,                     line_dash='dash', line_color='black',   row=1, col=1)
fig.add_hline(y= 2*residuos_arr.std(), line_dash='dot',  line_color='gray',    row=1, col=1)
fig.add_hline(y=-2*residuos_arr.std(), line_dash='dot',  line_color='gray',    row=1, col=1)

for lag_r in range(len(acf_resid)):
    bc = 'crimson' if abs(acf_resid[lag_r]) > ci_r else '#4CAF50'
    fig.add_trace(go.Scatter(x=[lag_r, lag_r], y=[0, acf_resid[lag_r]],
        mode='lines', line=dict(color=bc, width=3), showlegend=False), row=1, col=2)
fig.add_hline(y= ci_r, line_dash='dash', line_color='gray', row=1, col=2)
fig.add_hline(y=-ci_r, line_dash='dash', line_color='gray', row=1, col=2)

fig.update_layout(height=380, template='plotly_white', showlegend=False,
    title='Diagnóstico de Residuos — WindowAverage<br>'
          '<sup>Verde = no significativo (ruido) | Rojo = patrón remanente</sup>')
fig.show()

Residuos — media: -33.88  |  std: 110.51
Esperado en ruido blanco: media ≈ 0, desviación estándar constante

LJUNG-BOX SOBRE RESIDUOS DEL WindowAverage:
──────────────────────────────────────────────────────────
  Lag  1: p=0.0000  →  Quedan patrones ⚠️  → margen de mejora
  Lag  6: p=0.0000  →  Quedan patrones ⚠️  → margen de mejora
  Lag 12: p=0.0000  →  Quedan patrones ⚠️  → margen de mejora


In [23]:
# ── Instanciar y ajustar todos los modelos en una sola llamada ──────────
h = len(test)   # horizonte = longitud del conjunto de prueba

sf = StatsForecast(
    models=[
        Naive(),                        # ŷ = y_t (último valor)
        SeasonalNaive(season_length=52), # ŷ = y_{t-52} (misma semana año anterior)
        WindowAverage(window_size=3),    # ŷ = promedio últimas 3 obs.
        RandomWalkWithDrift()            # ŷ = y_t + drift (tendencia histórica)
    ],
    freq='W-MON'    # frecuencia semanal anclada al Lunes
)

sf.fit(train)
preds = sf.predict(h=h)

print("Pronósticos generados:")
print(f"  Modelos: {[c for c in preds.columns if c not in ['unique_id','ds']]}")
print(f"  Horizonte: {h} semanas")
preds.head()

Pronósticos generados:
  Modelos: ['Naive', 'SeasonalNaive', 'WindowAverage', 'RWD']
  Horizonte: 221 semanas


,unique_id,ds,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,2020-01-06,162.0,12.0,141.666667,162.287356
1,dengue_cali,2020-01-13,162.0,11.0,141.666667,162.574713
2,dengue_cali,2020-01-20,162.0,3.0,141.666667,162.862069
3,dengue_cali,2020-01-27,162.0,5.0,141.666667,163.149425
4,dengue_cali,2020-02-03,162.0,9.0,141.666667,163.436782


##SeasonalNieve predice muchos más casos

---
## 🔄 PARTE 9: Time Series Cross-Validation

El split simple de train/test tiene alta varianza — depende de qué período específico
cayó en el test. La **validación cruzada temporal** genera múltiples ventanas de evaluación,
produciendo métricas más confiables.


In [24]:
# ── Cross-Validation con StatsForecast ───────────────────────────────────
# n_windows=3: 3 folds de validación
# h=12:        horizonte de 12 meses en cada fold
# step_size=12: los folds se desplazan de año en año

cv_results = sf.cross_validation(
    df=df,
    h=52,
    n_windows=3,
    step_size=52
)

print(f"Resultados CV: {cv_results.shape[0]} filas × {cv_results.shape[1]} columnas")
print(f"Folds (cutoffs): {cv_results.cutoff.unique().tolist()}")
cv_results.head(6)

Resultados CV: 156 filas × 8 columnas
Folds (cutoffs): [Timestamp('2021-03-29 00:00:00'), Timestamp('2022-03-28 00:00:00'), Timestamp('2023-03-27 00:00:00')]


,unique_id,ds,cutoff,y,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,2021-04-05,2021-03-29,60.0,69.0,209.0,78.0,69.097104
1,dengue_cali,2021-04-12,2021-03-29,77.0,69.0,177.0,78.0,69.194208
2,dengue_cali,2021-04-19,2021-03-29,51.0,69.0,161.0,78.0,69.291312
3,dengue_cali,2021-04-26,2021-03-29,46.0,69.0,142.0,78.0,69.388416
4,dengue_cali,2021-05-03,2021-03-29,39.0,69.0,155.0,78.0,69.485520
5,dengue_cali,2021-05-10,2021-03-29,53.0,69.0,136.0,78.0,69.582624


In [25]:
# ── Métricas promedio por fold ────────────────────────────────────────────
# cv_results ya tiene columna 'y' — sin necesidad de rename
# Verificar columnas disponibles antes de evaluar
cv_models = [c for c in modelos_col if c in cv_results.columns]
cv_eval = evaluate(
    cv_results,
    metrics=[mae, rmse, smape],
    models=cv_models,
    target_col='y'
)

print("Métricas promedio en Cross-Validation (3 folds):")
print(cv_eval.to_string(index=False))

# ── Visualización de predicciones por fold ────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.ds, y=df.y, mode='lines',
                          name='Serie real', line=dict(color='black', width=1.5)))

colores_cv = ['#FF5722', '#4CAF50', '#9C27B0']
for i, cutoff in enumerate(sorted(cv_results.cutoff.unique())):
    fold = cv_results[cv_results.cutoff == cutoff]
    # Solo SeasonalNaive (mejor modelo)
    fig.add_trace(go.Scatter(
        x=fold.ds, y=fold.SeasonalNaive, mode='lines+markers',
        name=f'SeasonalNaive Fold {i+1} (cutoff {cutoff.strftime("%Y-%m")})',
        line=dict(color=colores_cv[i], width=3, dash='dot'),
        marker=dict(size=2)
    ))
    cutoff_str = cutoff.strftime('%Y-%m-%d')
    fig.add_shape(type='line', xref='x', yref='paper',
                  x0=cutoff_str, x1=cutoff_str, y0=0, y1=1,
                  line=dict(dash='dash', color=colores_cv[i], width=1.5),
                  opacity=0.4)

fig.update_layout(
    title='Time Series Cross-Validation — SeasonalNaive (3 folds)',
    xaxis_title='Fecha', yaxis_title='Ventas',
    height=420, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

Métricas promedio en Cross-Validation (3 folds):
  unique_id     cutoff metric      Naive  SeasonalNaive  WindowAverage        RWD
dengue_cali 2021-03-29    mae  15.769231      39.192308      22.884615  17.873804
dengue_cali 2022-03-28    mae  13.961538      22.730769      11.000000  14.853738
dengue_cali 2023-03-27    mae 155.210165     177.056319     160.479396 154.377926
dengue_cali 2021-03-29   rmse  18.828170      54.814021      25.780881  21.153887
dengue_cali 2022-03-28   rmse  15.675802      27.270723      12.656035  16.732381
dengue_cali 2023-03-27   rmse 199.722642     225.082791     206.283226 198.275011
dengue_cali 2021-03-29  smape   0.135143       0.237494       0.181336   0.149228
dengue_cali 2022-03-28  smape   0.176543       0.240309       0.145776   0.185040
dengue_cali 2023-03-27  smape   0.525600       0.663714       0.557043   0.520142


### 🔍 Interpretación — Cross-Validation

- Cada fold usa un período diferente de entrenamiento y prueba
- Las métricas promedio a través de los 3 folds son **más robustas** que una sola evaluación
- Si el modelo es consistente, sus métricas deberían ser similares en todos los folds
- Si hay mucha varianza entre folds, puede indicar inestabilidad o cambios en la serie

> 🔑 `StatsForecast.cross_validation()` maneja automáticamente el manejo temporal
> — no hay riesgo de data leakage.


In [26]:
# ── EJERCICIO 3: Ljung-Box sobre residuos del SeasonalNaive ──────────────
# TU CÓDIGO AQUÍ
# residuos = test_preds['y'] - test_preds['SeasonalNaive']
# result_resid = acorr_ljungbox(residuos, lags=[1, 6, 12], return_df=True)
# print(result_resid)
# Interpretación: ¿quedan patrones?
